In [ ]:
# %% [markdown]
# ### Proposed supplier-identification evaluation
# Benchmark integrations are retained below for reference but disabled by default.

# %%
from pathlib import Path
import sys
import pandas as pd

WORKING_DIR = Path.cwd()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / "code" / "01_supplier_identification").is_dir() else WORKING_DIR.parents[1]
IDENTIFICATION_CODE_DIR = PROJECT_ROOT / "code" / "01_supplier_identification"
sys.path.insert(0, str(IDENTIFICATION_CODE_DIR))

# -----------------------------
# Retriever imports
# -----------------------------
# BENCHMARK (disabled): from product_type_baseline_retriever import ProductTypeOnlyRetriever
# BENCHMARK (disabled): from histogram_baseline_retriever import Baseline2HistogramRetriever
from proposed_supplier_retriever import MultiLabelSupplierRetriever

RUN_BENCHMARKS = False  # BENCHMARK evaluations are disabled for this release.

# -----------------------------
# Common paths
# -----------------------------
BASE = PROJECT_ROOT
RESULT_DIR = BASE / "data" / "01_supplier_identification" / "main_split_70_30"
EPOCH = 10

# Proposed model input (also used by Benchmark 1 when enabled)
TRAIN_CSV = RESULT_DIR / "train_embeddings_epoch_010_with_metadata.csv"
TEST_CSV  = RESULT_DIR / "test_embeddings_epoch_010_with_metadata.csv"

# BENCHMARK 2 input paths (disabled)
HIST_DIR = RESULT_DIR / "histogram_baseline"
TRAIN_CSV_B2 = HIST_DIR / "train_histogram_features.csv"
TEST_CSV_B2  = HIST_DIR / "test_histogram_features.csv"

# -----------------------------
# Settings
# -----------------------------
K_THR = 11
B2_K_FOR_THR = 12
B2_SHAPE_SCALE = 10.0

INCLUDE_EXTRA_METRICS = True

# -----------------------------
# Utility: metrics -> row dict
# -----------------------------
def pack_metrics(model_name: str, metrics: dict, include_extra: bool = True) -> dict:
    row = {
        "Model": model_name,
        "AvgPrecision": float(metrics.get("AvgPrecision", float("nan"))),
        "AvgRecall": float(metrics.get("AvgRecall", float("nan"))),
        "AvgF1": float(metrics.get("AvgF1", float("nan"))),
        "AvgAccuracy": float(metrics.get("AvgAccuracy", float("nan"))),
    }
    if include_extra:
        for k in ["MacroPurity@τ", "MicroPurity@τ", "ViolationRate_any@τ", "WrongProduct@1@τ"]:
            if k in metrics:
                row[k] = float(metrics[k])
    return row

# -----------------------------
# Containers
# -----------------------------
rows = []

# optional reusable variables
retriever_b1 = None
retriever_b2 = None
retriever_m3 = None

train_df_b1 = test_df_b1 = feasible_suppliers_b1 = test_info_b1 = None
train_df_b2 = test_df_b2 = feasible_suppliers_b2 = test_info_b2 = None
train_df_m3 = test_df_m3 = feasible_suppliers_m3 = test_info_m3 = None

# -----------------------------
# BENCHMARK 1: Product-type retrieval (disabled)
# -----------------------------
if RUN_BENCHMARKS and TRAIN_CSV.exists() and TEST_CSV.exists():
    print(f"[INFO] Running Baseline1 using:\n  {TRAIN_CSV}\n  {TEST_CSV}")
    try:
        retriever_b1 = ProductTypeOnlyRetriever(
            train_csv=str(TRAIN_CSV),
            test_csv=str(TEST_CSV),
            k_for_threshold=None,
            require_same_component=False,
            require_same_assembly=False,
        )
        pred_sets_b1, metrics_b1 = retriever_b1.run_retrieval(verbose=False)

        train_df_b1 = pd.read_csv(TRAIN_CSV)
        test_df_b1  = pd.read_csv(TEST_CSV)
        feasible_suppliers_b1 = retriever_b1.get_feasible_suppliers()
        test_info_b1 = retriever_b1.get_test_info_dataframe()

        rows.append(
            pack_metrics("Model 1: Baseline1 (ProductTypeOnly)", metrics_b1, include_extra=INCLUDE_EXTRA_METRICS)
        )
    except Exception as e:
        print(f"[WARN] Baseline1 failed and will be skipped: {e}")
else:
    print("[INFO] BENCHMARK 1 evaluation is disabled.")

# -----------------------------
# BENCHMARK 2: Histogram retrieval (disabled)
# -----------------------------
if RUN_BENCHMARKS and TRAIN_CSV_B2.exists() and TEST_CSV_B2.exists():
    print(f"[INFO] Running Baseline2 using:\n  {TRAIN_CSV_B2}\n  {TEST_CSV_B2}")
    try:
        retriever_b2 = Baseline2HistogramRetriever(
            train_csv=str(TRAIN_CSV_B2),
            test_csv=str(TEST_CSV_B2),
            k_for_thr=B2_K_FOR_THR,
            metric_cols=None,
            shape_scale=B2_SHAPE_SCALE,
        )
        pred_sets_b2, metrics_b2 = retriever_b2.run_retrieval(
            verbose=False,
            require_same_component=False,
            require_same_assembly=False,
            log_top=0,
            margin_delta=0.01,
            debug_every=10**9,
            max_debug=0,
        )

        train_df_b2 = retriever_b2.train_df
        test_df_b2  = retriever_b2.get_test_info_dataframe()
        feasible_suppliers_b2 = retriever_b2.get_feasible_suppliers()
        test_info_b2 = retriever_b2.get_test_info_dataframe()

        rows.append(
            pack_metrics("Model 2: Baseline2 (Histogram)", metrics_b2, include_extra=INCLUDE_EXTRA_METRICS)
        )
    except Exception as e:
        print(f"[WARN] Baseline2 failed and will be skipped: {e}")
else:
    print("[INFO] BENCHMARK evaluations are disabled.")

# -----------------------------
# Run Model 3: Proposed
# -----------------------------
if TRAIN_CSV.exists() and TEST_CSV.exists():
    print(f"[INFO] Running Proposed model using:\n  {TRAIN_CSV}\n  {TEST_CSV}")
    try:
        retriever_m3 = MultiLabelSupplierRetriever(
            train_csv=TRAIN_CSV,
            test_csv=TEST_CSV,
            k_for_threshold=K_THR,
            require_same_component=False,
            require_same_assembly=False,
        )
        pred_sets_m3, metrics_m3 = retriever_m3.run_retrieval(verbose=False)

        train_df_m3 = pd.read_csv(TRAIN_CSV)
        test_df_m3  = pd.read_csv(TEST_CSV)
        feasible_suppliers_m3 = retriever_m3.get_feasible_suppliers()
        test_info_m3 = retriever_m3.get_test_info_dataframe()

        rows.append(
            pack_metrics("Model 3: Proposed (MultiLabel)", metrics_m3, include_extra=INCLUDE_EXTRA_METRICS)
        )
    except Exception as e:
        print(f"[WARN] Proposed model failed and will be skipped: {e}")
else:
    print("[WARN] Proposed model skipped because embedding CSVs do not exist.")

# -----------------------------
# Summary table
# -----------------------------
if len(rows) == 0:
    print("[WARN] No models were successfully evaluated.")
else:
    summary_df = pd.DataFrame(rows)

    core_cols = ["Model", "AvgPrecision", "AvgRecall", "AvgF1", "AvgAccuracy"]
    extra_cols = [c for c in summary_df.columns if c not in core_cols]
    summary_df = summary_df[core_cols + extra_cols]

    display(summary_df.style.format(precision=4))

# -----------------------------
# Optional: quick sanity prints
# -----------------------------
sanity_rows = []
if feasible_suppliers_b1 is not None:
    sanity_rows.append({"Model": "Baseline1", "NumQueries": len(feasible_suppliers_b1)})
if feasible_suppliers_b2 is not None:
    sanity_rows.append({"Model": "Baseline2", "NumQueries": len(feasible_suppliers_b2)})
if feasible_suppliers_m3 is not None:
    sanity_rows.append({"Model": "Proposed", "NumQueries": len(feasible_suppliers_m3)})

if len(sanity_rows) > 0:
    display(pd.DataFrame(sanity_rows))